# semcer quickstart

Score OCR output against a reference, on the free Colab GPU. Choose **Runtime > Change runtime type > T4 GPU** first.

In [ ]:
!pip install -q semcer

## One line

The first call downloads the model (3 GB).

In [ ]:
import semcer

result = semcer.score(reference="The total is 15 dollars.", hypothesis="The total is 75 dollars.")
print(result.explain())

## Several lines

`SemCER()` uses the GPU when there is one.

In [ ]:
references = [
    "Invoice No. 2024-117, due 3 March.",
    "Tổng cộng là 15 đồng.",
    "他昨天没有来，数字是12345。",
]
hypotheses = [
    "Invoice No. 2024-177, due 3 March",
    "Tong cộng là 15 đồng.",
    "他昨天没有来，数字是12845。",
]

scorer = semcer.SemCER()
results = scorer.score_batch(references=references, hypotheses=hypotheses)
for r in results:
    print(r.explain(), end="\n\n")

## The worst errors in a line

In [ ]:
for edit in results[0].worst_edits(3):
    print(edit.op, repr(edit.ref_grapheme), "->", repr(edit.hyp_grapheme), round(edit.severity, 2), edit.level)

## A file

One JSON object per line with `id`, `reference`, `hypothesis`.

In [ ]:
import json

with open("pairs.jsonl", "w", encoding="utf-8") as f:
    for n, (ref, hyp) in enumerate(zip(references, hypotheses)):
        f.write(json.dumps({"id": n, "reference": ref, "hypothesis": hyp}, ensure_ascii=False) + "\n")

!semcer score --input pairs.jsonl --output scores.jsonl --summary summary.json
!cat summary.json

More in the [usage guide](https://github.com/nguyenx2t/semcer/blob/main/docs/USAGE.md).